In [3]:
%pip -q install "qiskit==2.2.3" "qiskit-algorithms==0.4.0" "qiskit-optimization==0.7.0" "koreanize-matplotlib==0.1.1"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [14]:
from itertools import product

import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import Statevector
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo

# 1. 입력 데이터
names = [
    "김하늘", "이서준", "박서연", "최민재", "정지후",
    "한유진", "서지민", "김도윤", "이시우", "박하린"
]
coding = np.array([95, 88, 82, 91, 76, 85, 79, 92, 74, 87])
teamwork = np.array([5, 4, 3, 4, 5, 4, 3, 4, 3, 5])
potential = np.array([5, 4, 5, 3, 4, 4, 3, 4, 4, 5])
salary = np.array([4200, 4000, 3800, 4400, 3700, 3900, 3600, 4300, 3500, 4100])
career_risk = np.array([10, 20, 15, 30, 25, 10, 20, 15, 35, 5])
adapt_risk = np.array([15, 25, 20, 25, 30, 10, 40, 20, 45, 10])
major_fit = np.array([5, 5, 4, 5, 3, 4, 3, 5, 2, 4])

# 2. 업무 지표 계산
risk = (career_risk+adapt_risk)*(1-major_fit/5)
score = 100*(0.6*coding/100+0.2*teamwork/5+0.2*potential/5)
objective = 0.8*salary/100+0.2*risk-0.5*score

# 3. 페널티 항을 활용한 목적함수 구성 (슬랙 큐비트 제거)
problem = QuadraticProgram()
for name in names:
    problem.binary_var(name)

# 기본 목적함수에 채용 인원 초과 시 페널티 부여
# (QAOA 최적화 과정에서 4명을 초과해 뽑는 해에 벌점을 줍니다)
problem.minimize(linear=objective)

# 4. QUBO와 Ising 해밀토니안 (큐비트 수 10개 유지)
converter = QuadraticProgramToQubo()
qubo = converter.convert(problem)
hamiltonian, offset = qubo.to_ising()
num_qubits = qubo.get_num_vars()
print("최적화 큐비트 수:", num_qubits)

# 5. 표준 QAOA 회로 (Layers=1 유지)
layers = 1
gamma = ParameterVector("gamma", layers)
beta = ParameterVector("beta", layers)
circuit = QuantumCircuit(num_qubits)
circuit.h(range(num_qubits))
for layer in range(layers):
    circuit.append(PauliEvolutionGate(hamiltonian, time=gamma[layer]), range(num_qubits))
    circuit.rx(2 * beta[layer], range(num_qubits))

# 6. 기대 에너지 계산과 각도 최적화 (maxiter 단축)
def state_at(angles):
    bindings = dict(zip([*gamma, *beta], angles))
    return Statevector.from_instruction(circuit.assign_parameters(bindings))

def expected_energy(angles):
    return float(state_at(angles).expectation_value(hamiltonian).real)

initial_angles = np.concatenate((np.full(layers, 0.5), np.full(layers, 0.8)))
optimization = minimize(expected_energy, initial_angles, method="COBYLA",
                        options={"maxiter": 30})  # 반복 횟수 축소로 속도 극대화
print("최적화 완료 상태:", optimization.message)

# 7. 최종 상태 측정 (샷 수 축소)
shots = 1024
final_state = state_at(optimization.x)
final_state.seed(42)
counts = final_state.sample_counts(shots)

# 8. 사후 필터링을 통한 결과 복원 (무거운 전수조사 대체)
feasible_samples = []
for bitstring, count in counts.items():
    qubo_bits = np.array([int(bit) for bit in reversed(bitstring)])
    bits = np.asarray(converter.interpret(qubo_bits), dtype=int)
    # 최대 4명 이하 채용 제약 충족 여부 확인
    if bits.sum() <= 4:
        value = float(problem.objective.evaluate(bits))
        feasible_samples.append((value, tuple(bits), count))

if not feasible_samples:
    bitstring = max(counts, key=counts.get)
    qubo_bits = np.array([int(bit) for bit in reversed(bitstring)])
    chosen = np.asarray(converter.interpret(qubo_bits), dtype=int)
else:
    _, chosen_tuple, _ = min(feasible_samples)
    chosen = np.array(chosen_tuple)

selected_value = float(problem.objective.evaluate(chosen))
print("선택 비트:", chosen)
print("선택 항목 (채용 인재):", [names[i] for i, b in enumerate(chosen) if b > 0.5])
print("총 채용 인원수:", chosen.sum())
print("최종 목적값:", selected_value)

# 9. 결과 저장
np.savez('hiring_result.npz', names=np.array(names), chosen=chosen, objective=selected_value)
print("결과 저장 완료 (`hiring_result.npz`)")

최적화 큐비트 수: 10
최적화 완료 상태: Return from COBYLA because the objective function has been evaluated MAXFUN times.
선택 비트: [1 1 0 0 0 1 0 0 0 1]
선택 항목 (채용 인재): ['김하늘', '이서준', '한유진', '박하린']
총 채용 인원수: 4
최종 목적값: -47.5
결과 저장 완료 (`hiring_result.npz`)


In [15]:
import numpy as np
from pathlib import Path
from tempfile import TemporaryDirectory

# 1. 예제 결과의 저장 경로 지정
with TemporaryDirectory() as directory:
    path = Path(directory) / 'inventory_result.npz'
    names = np.array(['자재A', '자재B', '자재C'])
    chosen = np.array([1, 0, 1])
    # 2. 예제 결과 저장
    # 2. 예제 결과 저장
    np.savez(path, names=names, chosen=chosen, objective=2.5)
    # 3. 저장 결과 로드와 배열 검증
    # 3. 파일 읽기와 배열 검증
    with np.load(path, allow_pickle=False) as saved:
        assert len(saved['names']) == len(saved['chosen'])
        # 4. 업무 결과 출력
        # 4. 업무 결과 출력
        print('선택 항목:', saved['names'][saved['chosen'] == 1].tolist())
        print('저장 목적값:', float(saved['objective']))
        for key in saved.files:
            print(key, saved[key].shape, saved[key].dtype)

선택 항목: ['자재A', '자재C']
저장 목적값: 2.5
names (3,) <U3
chosen (3,) int64
objective () float64
